# 🏗️ Modelado Dimensional e Implementación del Pipeline ETL en Python

### Autor: Néstor León | Business Intelligence & Analytics
---

Una base de datos de producción normalizada (3FN) es perfecta para transacciones bancarias o cajas de cobro, pero es pésima para responder preguntas como *"¿cuánto vendimos por región y trimestre frente al año anterior?"*, porque requiere cruzar 15 tablas con JOINs pesadísimos.

Siguiendo el estándar de **Ralph Kimball**, en este notebook:
1. Diseñamos un **Esquema en Estrella (Star Schema)** conectando una tabla de hechos central (`fact_sales`) con dimensiones descriptivas (`dim_time`, `dim_customer`, `dim_store`).
2. Implementamos el pipeline **ETL (Extract, Transform, Load)** en Python con `pandas` y `DuckDB`.
3. Validamos la consistencia relacional y creamos la base analítica.


In [ ]:
import pandas as pd
import numpy as np
import duckdb
import os

print("Entorno de ingeniería de datos configurado con DuckDB.")


## 1. Fase de Extracción: Fuentes Heterogéneas

Leemos las fuentes originales de ventas (`SSSALES.xlsx` y maestros de dimensiones). Si corremos en un entorno donde no esté el fichero Excel, generamos el juego de datos exacto de forma completamente autónoma.


In [ ]:
# Simulación o lectura de las hojas transaccionales
data_folder = 'data'
sales_file = os.path.join(data_folder, 'SSSALES.xlsx')

if os.path.exists(sales_file):
    print(f"Cargando datos reales desde {sales_file}...")
    df_sales_raw = pd.read_excel(sales_file)
else:
    print("Generando datos transaccionales para pipeline reproducible...")
    np.random.seed(42)
    n_rows = 1500
    df_sales_raw = pd.DataFrame({
        'TransactionID': range(1001, 1001 + n_rows),
        'CustID': np.random.choice(['CUST_01', 'CUST_02', 'CUST_03', 'CUST_04', 'CUST_05', 'CUST_06'], n_rows),
        'StoreID': np.random.choice(['STR_MAD', 'STR_BCN', 'STR_VAL', 'STR_SEV'], n_rows),
        'Date': pd.date_range('2023-01-01', '2023-12-31', periods=n_rows),
        'Quantity': np.random.randint(1, 10, n_rows),
        'UnitPrice': np.random.choice([15.50, 45.00, 120.00, 250.00, 89.90], n_rows)
    })
    df_sales_raw['TotalSales'] = df_sales_raw['Quantity'] * df_sales_raw['UnitPrice']

df_sales_raw.head()


## 2. Fase de Transformación: Construcción de Dimensiones y Claves Subrogadas

Una regla de oro en Data Warehousing: **nunca uses claves de negocio originales como clave primaria en el almacén**. Generamos claves subrogadas numéricas enteras (*Surrogate Keys*) que garantizan independencia frente a cambios en los sistemas transaccionales origen.


In [ ]:
# 1. Dimensión Tiempo
fechas_unicas = pd.to_datetime(df_sales_raw['Date']).dt.normalize().drop_duplicates().sort_values()
dim_time = pd.DataFrame({
    'time_key': fechas_unicas.dt.strftime('%Y%m%d').astype(int),
    'date_val': fechas_unicas,
    'day_number': fechas_unicas.dt.day,
    'month_number': fechas_unicas.dt.month,
    'month_name': fechas_unicas.dt.strftime('%B'),
    'quarter': fechas_unicas.dt.quarter,
    'year_number': fechas_unicas.dt.year
})

# 2. Dimensión Cliente
clientes_maestro = pd.DataFrame({
    'customer_key': [1, 2, 3, 4, 5, 6],
    'customer_id': ['CUST_01', 'CUST_02', 'CUST_03', 'CUST_04', 'CUST_05', 'CUST_06'],
    'customer_name': ['Iberdrola S.A.', 'Mercadona Logística', 'Inditex Retail', 'Telefónica Móviles', 'Banco Santander', 'Repsol Energía'],
    'city': ['Bilbao', 'Valencia', 'A Coruña', 'Madrid', 'Santander', 'Madrid'],
    'state': ['Vizcaya', 'Valencia', 'A Coruña', 'Madrid', 'Cantabria', 'Madrid']
})

# 3. Dimensión Tienda / Sucursal
tiendas_maestro = pd.DataFrame({
    'store_key': [101, 102, 103, 104],
    'store_id': ['STR_MAD', 'STR_BCN', 'STR_VAL', 'STR_SEV'],
    'store_name': ['Sucursal Madrid Castellana', 'Sucursal Barcelona Diagonal', 'Sucursal Valencia Colón', 'Sucursal Sevilla Nervión'],
    'division_name': ['División Centro', 'División Noreste', 'División Este', 'División Sur'],
    'region': ['Centro', 'Cataluña', 'Comunidad Valenciana', 'Andalucía']
})

print("Dimensiones construidas con claves subrogadas.")
dim_time.head(3)


### Construcción de la Tabla de Hechos (Fact Sales)

Cruzamos las transacciones con las dimensiones para mapear las claves de negocio hacia las *Surrogate Keys*.


In [ ]:
fact_sales = df_sales_raw.copy()

# Mapeamos time_key
fact_sales['time_key'] = pd.to_datetime(fact_sales['Date']).dt.strftime('%Y%m%d').astype(int)

# Mapeamos customer_key
cust_map = dict(zip(clientes_maestro['customer_id'], clientes_maestro['customer_key']))
fact_sales['customer_key'] = fact_sales['CustID'].map(cust_map)

# Mapeamos store_key
store_map = dict(zip(tiendas_maestro['store_id'], tiendas_maestro['store_key']))
fact_sales['store_key'] = fact_sales['StoreID'].map(store_map)

# Seleccionamos columnas finales de hechos y medidas numéricas aditivas
fact_sales_final = fact_sales[[
    'TransactionID', 'time_key', 'customer_key', 'store_key', 'Quantity', 'UnitPrice', 'TotalSales'
]].rename(columns={
    'TransactionID': 'sales_id',
    'Quantity': 'quantity_sold',
    'UnitPrice': 'unit_price',
    'TotalSales': 'total_sales_amount'
})

fact_sales_final.head()


## 3. Fase de Carga: Base de Datos Analítica Embebida (DuckDB)

Persistimos el modelo dimensional completo en una base de datos DuckDB relacional ultrarrápida.


In [ ]:
con = duckdb.connect('sales_datawarehouse.duckdb')

con.execute("CREATE OR REPLACE TABLE dim_time AS SELECT * FROM dim_time")
con.execute("CREATE OR REPLACE TABLE dim_customer AS SELECT * FROM clientes_maestro")
con.execute("CREATE OR REPLACE TABLE dim_store AS SELECT * FROM tiendas_maestro")
con.execute("CREATE OR REPLACE TABLE fact_sales AS SELECT * FROM fact_sales_final")

print("Data Warehouse poblado con éxito en DuckDB.")
# Verificamos número de registros cargados
con.execute("""
    SELECT 
        (SELECT count(*) FROM fact_sales) AS total_hechos,
        (SELECT count(*) FROM dim_customer) AS clientes,
        (SELECT count(*) FROM dim_store) AS tiendas,
        (SELECT count(*) FROM dim_time) AS fechas
""").df()


## 4. Conclusiones

1. **Diseño Desacoplado y Robusto**: Separar las entidades en hechos y dimensiones permite añadir nuevas métricas (descuentos, devoluciones) o dimensiones (productos, comerciales) sin reescribir la base del sistema.
2. **Listo para Analítica de Alto Rendimiento**: Con las tablas almacenadas en columnar, las consultas agregativas y los dashboards responderán en milisegundos.
